# Final results and analysis

North star: produce reproducible, language-pack-first grounded QA evidence with citation-aware outputs and transparent trust signals.

This notebook consolidates evaluation and ablation artifacts into analysis tables that can be reused for reporting.

### Reader guide
1. Start with KPI and ablation sections to understand baseline behavior and trust-layer sensitivity.
2. Use reproducibility diagnostics to verify whether comparisons are made under aligned config hashes.
3. Use finetune summary and deltas to judge whether a variant meaningfully improves grounded behavior over baseline.

In [11]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [12]:
import importlib.util
import sys
from subprocess import check_call

def _ensure(pkg: str, import_name: str | None = None) -> None:
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print(f"Installing {pkg}...")
        check_call([sys.executable, "-m", "pip", "install", pkg])
    else:
        print(f"{pkg} already installed")

for pkg, mod in [("duckdb", "duckdb"), ("pandas", "pandas"), ("pyarrow", "pyarrow")]:
    _ensure(pkg, mod)

duckdb already installed
pandas already installed
pyarrow already installed


In [13]:
from pathlib import Path
import os
import shutil

def _find_repo_root() -> Path:
    env_root = os.getenv("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    for cand in candidates:
        if (cand / "pyproject.toml").exists() and (cand / "src").exists():
            return cand

    return cwd

repo_root = _find_repo_root()
os.environ["PROJECT_ROOT"] = str(repo_root)
print("PROJECT_ROOT set to", repo_root)

dst = repo_root / "artifacts"
dst.mkdir(parents=True, exist_ok=True)

# Optional escape hatch: manually point to an external artifacts directory.
# If PGQA_ARTIFACTS_DIR points at either <dir>/artifacts or <dir>, we merge it.
external_artifacts_dir = os.getenv("PGQA_ARTIFACTS_DIR")
if external_artifacts_dir:
    src = Path(external_artifacts_dir).expanduser().resolve()
    if src.exists():
        src_effective = src / "artifacts" if (src / "artifacts").exists() else src
        shutil.copytree(src_effective, dst, dirs_exist_ok=True)
        print("Merged artifacts from PGQA_ARTIFACTS_DIR:", src_effective)
    else:
        print("PGQA_ARTIFACTS_DIR does not exist:", src)

expected = [
    dst / "runs" / "eval_results.parquet",
    dst / "tables" / "ablation_results.parquet",
    dst / "tables" / "finetune_eval_summary.parquet",
    dst / "tables" / "finetune_variant_leaderboard.parquet",
]
missing = [p for p in expected if not p.exists()]

print("Artifacts destination now contains:", [p.name for p in sorted(dst.iterdir())])
if missing:
    print("Missing expected artifact files:")
    for p in missing:
        print(" -", p.relative_to(repo_root))
    print("Tip: export/copy artifacts locally or set PGQA_ARTIFACTS_DIR and rerun this cell.")
else:
    print("All expected core artifacts are present locally.")

PROJECT_ROOT set to /Users/aaliyan/aaliyan/polyglot-grounded-qa
Artifacts destination now contains: ['.DS_Store', 'figures', 'indexes', 'runs', 'tables']
All expected core artifacts are present locally.


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sys

import duckdb
import pandas as pd

root = Path(os.getenv("PROJECT_ROOT", Path.cwd().resolve()))
if not (root / "artifacts").exists() and (root.parent / "artifacts").exists():
    root = root.parent

eval_path = root / "artifacts" / "runs" / "eval_results.parquet"
ablation_path = root / "artifacts" / "tables" / "ablation_results.parquet"
output_dir = root / "artifacts" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", root)
print("Eval exists:", eval_path.exists())
print("Ablation exists:", ablation_path.exists())
print("Tip: set PROJECT_ROOT if this notebook is not running at repo root.")

def _build_metadata(cfg, language: str) -> dict:
    config_json = json.dumps(cfg.model_dump(mode="json"), sort_keys=True)
    config_hash = hashlib.sha256(config_json.encode("utf-8")).hexdigest()
    return {
        "run_name": cfg.pipeline.run_name,
        "language": language,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "config_hash": config_hash,
    }

if not eval_path.exists() or not ablation_path.exists():
    print("Artifacts missing. Generating minimal eval/ablation in-place...")
    sys.path.insert(0, str(root / "src"))
    from polyglot_grounded_qa import create_default_pipeline
    from polyglot_grounded_qa.core.config_loader import load_app_config
    cfg = load_app_config(project_root=root)
    language = cfg.pipeline.default_language
    metadata = _build_metadata(cfg, language)
    pipeline = create_default_pipeline(str(root))

    eval_queries = [
        "What is language-pack architecture?",
        "Why are citations required in grounded QA?",
    ]
    eval_rows = []
    for query in eval_queries:
        result = pipeline.run(query=query, language=language)
        eval_rows.append(
            {
                **metadata,
                "query": query,
                "answer": result.answer,
                "abstained": result.abstained,
                "citation_count": len(result.citations),
            }
        )
    pd.DataFrame(eval_rows).to_parquet(eval_path, index=False)
    print("Wrote eval artifact:", eval_path)

    settings = [
        {"name": "baseline", "k": 10},
        {"name": "no_rerank", "k": 1},
    ]
    ablation_rows = []
    query = "How does locale inheritance work?"
    for setting in settings:
        pipeline.top_k_rerank = int(setting["k"])
        result = pipeline.run(query=query, language=language)
        ablation_rows.append(
            {
                **metadata,
                "variant": setting["name"],
                "answer": result.answer,
                "abstained": result.abstained,
                "citation_count": len(result.citations),
            }
        )
    pd.DataFrame(ablation_rows).to_parquet(ablation_path, index=False)
    print("Wrote ablation artifact:", ablation_path)

con = duckdb.connect()

Project root: /Users/aaliyan/aaliyan/polyglot-grounded-qa
Eval exists: True
Ablation exists: True
Tip: set PROJECT_ROOT if this notebook is not running at repo root.


## 1) Evaluation KPI summary

These metrics summarize run stability and answer behavior using the latest eval artifact.

In [15]:
eval_overview = con.execute(
    f"""
    SELECT
        run_name,
        language,
        config_hash,
        COUNT(*) AS question_count,
        SUM(CASE WHEN abstained THEN 1 ELSE 0 END) AS abstained_count,
        AVG(citation_count)::DOUBLE AS avg_citations
    FROM read_parquet('{eval_path}')
    GROUP BY run_name, language, config_hash
    ORDER BY run_name, language
    """
).df()

eval_overview['abstain_rate'] = (
    eval_overview['abstained_count'] / eval_overview['question_count']
).fillna(0.0)

eval_overview

,run_name,language,config_hash,question_count,abstained_count,avg_citations,abstain_rate
0,local-notebook-run,base,241522950ba1092e2fbfb1fb39fe34d311b3d24f5de2f8...,2,0.0,1.0,0.0


### KPI interpretation notes
- `abstain_rate` is a trust-control signal: higher can be better when uncertainty is real, but too high may indicate under-answering.
- `avg_citations` is only a structural proxy; quality is judged later by citation precision/recall in finetune evaluation.
- Compare rows by both `run_name` and `language` before concluding cross-run improvements.

## 2) Ablation comparison

Compare baseline vs no-rerank using abstention and citation behavior as trust-layer proxies.

In [16]:
ablation_summary = con.execute(
    f"""
    SELECT
        run_name,
        language,
        variant,
        COUNT(*) AS sample_count,
        SUM(CASE WHEN abstained THEN 1 ELSE 0 END) AS abstained_count,
        AVG(citation_count)::DOUBLE AS avg_citations
    FROM read_parquet('{ablation_path}')
    GROUP BY run_name, language, variant
    ORDER BY run_name, language, variant
    """
).df()

ablation_summary['abstain_rate'] = (
    ablation_summary['abstained_count'] / ablation_summary['sample_count']
).fillna(0.0)

pivot = ablation_summary.pivot(
    index=['run_name', 'language'],
    columns='variant',
    values=['abstain_rate', 'avg_citations'],
)

pivot.columns = [
    '_'.join([str(part) for part in col if str(part) != '']).strip('_')
    for col in pivot.columns.to_flat_index()
]

if 'abstain_rate_baseline' in pivot.columns and 'abstain_rate_no_rerank' in pivot.columns:
    pivot['delta_abstain_rate_no_rerank_minus_baseline'] = (
        pivot['abstain_rate_no_rerank'] - pivot['abstain_rate_baseline']
    )

if 'avg_citations_baseline' in pivot.columns and 'avg_citations_no_rerank' in pivot.columns:
    pivot['delta_avg_citations_no_rerank_minus_baseline'] = (
        pivot['avg_citations_no_rerank'] - pivot['avg_citations_baseline']
    )

ablation_summary, pivot

(             run_name language    variant  sample_count  abstained_count  \
 0  local-notebook-run     base   baseline             1              0.0   
 1  local-notebook-run     base  no_rerank             1              0.0   
 
    avg_citations  abstain_rate  
 0            1.0           0.0  
 1            1.0           0.0  ,
                              abstain_rate_baseline  abstain_rate_no_rerank  \
 run_name           language                                                  
 local-notebook-run base                        0.0                     0.0   
 
                              avg_citations_baseline  avg_citations_no_rerank  \
 run_name           language                                                    
 local-notebook-run base                         1.0                      1.0   
 
                              delta_abstain_rate_no_rerank_minus_baseline  \
 run_name           language                                                
 local-notebook-run base  

### Ablation interpretation notes
- If `delta_abstain_rate_no_rerank_minus_baseline > 0`, disabling reranking made the system abstain more often.
- If `delta_avg_citations_no_rerank_minus_baseline < 0`, reranking likely helps recover supporting evidence.
- Treat this section as directional evidence for pipeline choices, not a final quality verdict.

## 3) Reproducibility diagnostics and exports

Persist final tables for downstream reporting and confirm whether eval/ablation share the same config hash.

In [17]:
eval_hashes = set(
    con.execute(f"SELECT DISTINCT config_hash FROM read_parquet('{eval_path}')").fetchdf()['config_hash']
    .tolist()
)
ablation_hashes = set(
    con.execute(f"SELECT DISTINCT config_hash FROM read_parquet('{ablation_path}')").fetchdf()['config_hash']
    .tolist()
)

shared_hashes = eval_hashes.intersection(ablation_hashes)
hash_alignment = len(shared_hashes) > 0

eval_overview.to_parquet(output_dir / 'final_eval_overview.parquet', index=False)
ablation_summary.to_parquet(output_dir / 'final_ablation_summary.parquet', index=False)
pivot.to_parquet(output_dir / 'final_ablation_deltas.parquet', index=False)

diagnostics = pd.DataFrame(
    [
        {'metric': 'eval_config_hash_count', 'value': str(len(eval_hashes))},
        {'metric': 'ablation_config_hash_count', 'value': str(len(ablation_hashes))},
        {'metric': 'shared_config_hash_count', 'value': str(len(shared_hashes))},
        {'metric': 'config_hash_aligned', 'value': str(hash_alignment)},
    ]
)
diagnostics.to_parquet(output_dir / 'final_repro_diagnostics.parquet', index=False)

diagnostics

,metric,value
0,eval_config_hash_count,1
1,ablation_config_hash_count,1
2,shared_config_hash_count,1
3,config_hash_aligned,True


## 4) Finetune evaluation bridge

Load finetune-vs-baseline evaluation summaries produced by `scripts/run_finetune_eval.py`. This section is the direct comparison entrypoint for tuned adapters.

In [18]:
finetune_summary_path = root / 'artifacts' / 'tables' / 'finetune_eval_summary.parquet'
finetune_by_language_path = root / 'artifacts' / 'tables' / 'finetune_eval_by_language.parquet'

def _ensure_grounded_trust_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = [
        'abstain_accuracy',
        'avg_citation_precision',
        'avg_citation_recall',
        'avg_answer_token_f1',
    ]
    if all(col in out.columns for col in required):
        computed = (
            0.2 * out['abstain_accuracy'].astype(float)
            + 0.3 * out['avg_citation_precision'].astype(float)
            + 0.3 * out['avg_citation_recall'].astype(float)
            + 0.2 * out['avg_answer_token_f1'].astype(float)
        )
        if 'grounded_trust_score' not in out.columns:
            out['grounded_trust_score'] = computed
        else:
            out['grounded_trust_score'] = out['grounded_trust_score'].fillna(computed)
    return out

if finetune_summary_path.exists():
    finetune_summary = con.execute(
        f"SELECT * FROM read_parquet('{finetune_summary_path}') ORDER BY timestamp_utc DESC"
    ).df()
    finetune_summary = _ensure_grounded_trust_score(finetune_summary)
    print('Finetune summary available')
    display(finetune_summary)

    finetune_summary.to_parquet(output_dir / 'final_finetune_eval_summary.parquet', index=False)

    if finetune_by_language_path.exists():
        finetune_by_language = con.execute(
            f"SELECT * FROM read_parquet('{finetune_by_language_path}') ORDER BY language"
        ).df()
        finetune_by_language = _ensure_grounded_trust_score(finetune_by_language)
        display(finetune_by_language)
        finetune_by_language.to_parquet(
            output_dir / 'final_finetune_eval_by_language.parquet', index=False
        )
else:
    print('No finetune eval artifacts yet. Run scripts/run_finetune_eval.py first.')

Finetune summary available


,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,grounded_trust_score
0,tuned-adapter-v1,2026-04-03T18:35:19.563966+00:00,29,0.793103,0.206897,0.206897,0.000000,0.282759
1,grounded-heuristic-v1,2026-03-31T19:01:21.386827+00:00,29,1.000000,1.000000,1.000000,0.260780,0.852156
2,baseline-pipeline,2026-03-31T19:01:21.058167+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664
3,baseline-pipeline,2026-03-31T19:00:58.518759+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664
4,grounded-heuristic-v1,2026-03-31T18:59:13.676338+00:00,29,1.000000,1.000000,1.000000,0.260780,0.852156
5,baseline-pipeline,2026-03-31T18:59:13.366185+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664
6,grounded-heuristic-v1,2026-03-31T18:47:39.993277+00:00,29,1.000000,1.000000,1.000000,0.260780,0.852156
7,tuned-control-baseline,2026-03-31T18:44:23.558926+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664
8,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,29,1.000000,1.000000,1.000000,1.000000,1.000000
9,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664


,variant,timestamp_utc,language,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,grounded_trust_score
0,tuned-adapter-v1,2026-04-03T18:35:19.563966+00:00,base,6,0.666667,0.333333,0.333333,0.000000,0.333333
1,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667
2,grounded-heuristic-v1,2026-03-31T19:01:21.386827+00:00,base,6,1.000000,1.000000,1.000000,0.357092,0.871418
3,baseline-pipeline,2026-03-31T19:01:21.058167+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667
4,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,base,6,1.000000,1.000000,1.000000,1.000000,1.000000
5,baseline-pipeline,2026-03-31T19:00:58.518759+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667
6,tuned-control-baseline,2026-03-31T18:44:23.558926+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667
7,grounded-heuristic-v1,2026-03-31T18:59:13.676338+00:00,base,6,1.000000,1.000000,1.000000,0.357092,0.871418
8,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667
9,baseline-pipeline,2026-03-31T18:59:13.366185+00:00,base,6,0.666667,0.000000,0.000000,0.041667,0.141667


### Finetune bridge interpretation notes
- This table is a run history across variants; use `timestamp_utc` to identify the latest run for each variant.
- `oracle-upper-bound` is a ceiling diagnostic, not a deployable model.
- A variant is practically meaningful only if it improves grounding metrics while keeping answer quality acceptable.

## 5) Baseline vs tuned deltas

When multiple variants exist in `finetune_eval_summary.parquet`, compute deltas against the baseline variant and rank primarily by `delta_grounded_trust_score`.

In [19]:
if finetune_summary_path.exists():
    history = con.execute(
        f"SELECT * FROM read_parquet('{finetune_summary_path}') ORDER BY timestamp_utc DESC"
    ).df()
    history = _ensure_grounded_trust_score(history)

    baseline_rows = history[history['variant'] == 'baseline-pipeline']
    comparison_rows = history[history['variant'] != 'baseline-pipeline']

    if not baseline_rows.empty and not comparison_rows.empty:
        baseline = baseline_rows.iloc[0]
        latest_by_variant = comparison_rows.sort_values('timestamp_utc', ascending=False).groupby('variant').head(1)

        metric_cols = [
            'abstain_accuracy',
            'avg_citation_precision',
            'avg_citation_recall',
            'avg_answer_token_f1',
            'grounded_trust_score',
        ]
        metric_cols = [m for m in metric_cols if m in latest_by_variant.columns and m in baseline.index]

        deltas = latest_by_variant.copy()
        for metric in metric_cols:
            deltas[f'delta_{metric}'] = deltas[metric] - float(baseline[metric])

        cols = [
            'variant',
            'timestamp_utc',
            'rows',
            'abstain_accuracy',
            'avg_citation_precision',
            'avg_citation_recall',
            'avg_answer_token_f1',
            'grounded_trust_score',
            'delta_abstain_accuracy',
            'delta_avg_citation_precision',
            'delta_avg_citation_recall',
            'delta_avg_answer_token_f1',
            'delta_grounded_trust_score',
        ]
        cols = [c for c in cols if c in deltas.columns]
        deltas = deltas[cols].sort_values('delta_grounded_trust_score', ascending=False) if 'delta_grounded_trust_score' in cols else deltas[cols]
        display(deltas)
        deltas.to_parquet(output_dir / 'final_finetune_eval_deltas.parquet', index=False)
    else:
        print('Need both baseline and tuned variants to compute deltas.')
else:
    print('No finetune summary parquet found.')

,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,grounded_trust_score,delta_abstain_accuracy,delta_avg_citation_precision,delta_avg_citation_recall,delta_avg_answer_token_f1,delta_grounded_trust_score
8,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,29,1.000000,1.000000,1.000000,1.000000,1.000000,0.206897,1.000000,1.000000,0.964784,0.834336
1,grounded-heuristic-v1,2026-03-31T19:01:21.386827+00:00,29,1.000000,1.000000,1.000000,0.260780,0.852156,0.206897,1.000000,1.000000,0.225564,0.686492
0,tuned-adapter-v1,2026-04-03T18:35:19.563966+00:00,29,0.793103,0.206897,0.206897,0.000000,0.282759,0.000000,0.206897,0.206897,-0.035216,0.117095
7,tuned-control-baseline,2026-03-31T18:44:23.558926+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664,0.000000,0.000000,0.000000,0.000000,0.000000


## 6) Practical variant leaderboard

Use the exported leaderboard to make trust-gated model decisions reproducibly outside this notebook.

In [20]:
leaderboard_path = output_dir / 'finetune_variant_leaderboard.parquet'
leaderboard_md_path = output_dir / 'finetune_variant_leaderboard.md'

if leaderboard_path.exists():
    leaderboard = pd.read_parquet(leaderboard_path)
    display(leaderboard)

    practical_pass = leaderboard[
        (leaderboard['is_practical_variant'] == True)
        & (leaderboard['passes_grounding_gate'] == True)
    ]
    if not practical_pass.empty:
        print('Top practical gate-pass candidate:')
        display(practical_pass.head(1))
    else:
        print('No practical variant currently passes grounding gate.')
else:
    print('Leaderboard artifact not found. Re-run scripts/run_finetune_eval.py to generate it.')

,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,grounded_trust_score,delta_abstain_accuracy,delta_avg_citation_precision,delta_avg_citation_recall,delta_avg_answer_token_f1,delta_grounded_trust_score,is_practical_variant,passes_grounding_gate
0,grounded-heuristic-v1,2026-03-31T19:01:21.386827+00:00,29,1.000000,1.000000,1.000000,0.260780,0.852156,0.206897,1.000000,1.000000,0.225564,0.686492,True,True
1,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,29,1.000000,1.000000,1.000000,1.000000,1.000000,0.206897,1.000000,1.000000,0.964784,0.834336,False,False
2,tuned-adapter-v1,2026-04-03T18:35:19.563966+00:00,29,0.793103,0.206897,0.206897,0.000000,0.282759,0.000000,0.206897,0.206897,-0.035216,0.117095,True,False
3,tuned-control-baseline,2026-03-31T18:44:23.558926+00:00,29,0.793103,0.000000,0.000000,0.035216,0.165664,0.000000,0.000000,0.000000,0.000000,0.000000,False,False


Top practical gate-pass candidate:


,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,grounded_trust_score,delta_abstain_accuracy,delta_avg_citation_precision,delta_avg_citation_recall,delta_avg_answer_token_f1,delta_grounded_trust_score,is_practical_variant,passes_grounding_gate
0,grounded-heuristic-v1,2026-03-31T19:01:21.386827+00:00,29,1.0,1.0,1.0,0.26078,0.852156,0.206897,1.0,1.0,0.225564,0.686492,True,True


### Delta decision rubric
- Treat `oracle-upper-bound` as a feasibility signal, not a shipping candidate.
- Prioritize practical variants with positive `delta_grounded_trust_score` and non-negative answer quality deltas.
- Use citation precision/recall deltas as hard trust gates; use answer-token-F1 and trust score as ranking signals.

In [21]:
from pathlib import Path
from IPython.display import Markdown, display

takeaway_lines = [
    "## 7) Reader takeaway summary",
    "",
    "This section converts the latest delta table into plain-language findings.",
    "",
]

deltas_path = output_dir / 'final_finetune_eval_deltas.parquet'
leaderboard_path = output_dir / 'finetune_variant_leaderboard.parquet'

if deltas_path.exists():
    latest_deltas = pd.read_parquet(deltas_path)

    sort_col = 'delta_grounded_trust_score' if 'delta_grounded_trust_score' in latest_deltas.columns else 'delta_avg_answer_token_f1'
    latest_deltas = latest_deltas.sort_values(sort_col, ascending=False)

    if latest_deltas.empty:
        takeaway_lines.append("- No comparison variants are available yet.")
    else:
        best_overall = latest_deltas.iloc[0]
        best_overall_delta = float(best_overall.get(sort_col, 0.0))
        takeaway_lines.append(
            f"- Highest {sort_col} (including diagnostics): **{best_overall['variant']}** "
            f"($\\Delta$ = {best_overall_delta:.4f})."
        )

        practical_variant = None
        if leaderboard_path.exists():
            leaderboard = pd.read_parquet(leaderboard_path)
            gated = leaderboard[
                (leaderboard['is_practical_variant'] == True)
                & (leaderboard['passes_grounding_gate'] == True)
            ]
            if not gated.empty:
                practical_variant = gated.iloc[0]['variant']

        if practical_variant is not None:
            best_practical = latest_deltas[latest_deltas['variant'] == practical_variant].head(1)
            if not best_practical.empty:
                best_practical = best_practical.iloc[0]
                trust_delta = float(best_practical.get('delta_grounded_trust_score', 0.0))
                f1_delta = float(best_practical.get('delta_avg_answer_token_f1', 0.0))
                abstain_delta = float(best_practical.get('delta_abstain_accuracy', 0.0))
                cit_p_delta = float(best_practical.get('delta_avg_citation_precision', 0.0))
                cit_r_delta = float(best_practical.get('delta_avg_citation_recall', 0.0))
                takeaway_lines.append(
                    f"- Recommended practical variant today (leaderboard gate-pass): **{best_practical['variant']}** "
                    f"($\\Delta$ trust = {trust_delta:.4f}, $\\Delta$ F1 = {f1_delta:.4f}, "
                    f"$\\Delta$ abstain = {abstain_delta:.4f}, "
                    f"$\\Delta$ citation precision = {cit_p_delta:.4f}, "
                    f"$\\Delta$ citation recall = {cit_r_delta:.4f})."
                )
        else:
            non_oracle = latest_deltas[~latest_deltas['variant'].astype(str).str.contains('oracle', case=False, na=False)]
            if not non_oracle.empty:
                best_non_oracle = non_oracle.iloc[0]
                takeaway_lines.append(
                    f"- Best non-oracle variant available: **{best_non_oracle['variant']}**."
                )

        meaningful = latest_deltas[
            (latest_deltas.get('delta_avg_citation_precision', 0) > 0)
            & (latest_deltas.get('delta_avg_citation_recall', 0) > 0)
            & (latest_deltas.get('delta_avg_answer_token_f1', 0) > 0)
        ]
        if meaningful.empty:
            takeaway_lines.append(
                "- No variant currently improves citation precision, citation recall, and answer F1 simultaneously."
            )
        else:
            names = ', '.join(meaningful['variant'].tolist())
            takeaway_lines.append(
                f"- Variants with simultaneous gains on grounding and answer quality: {names}."
            )
else:
    takeaway_lines.append("- Delta artifact not found. Run section 5 first.")

takeaway_lines.extend(
    [
        "",
        "### Where we stand",
        "- The evaluation loop is reproducible end-to-end and now tracks a composite grounded trust score.",
        "- Practical ranking is now artifact-backed via finetune_variant_leaderboard outputs.",
        "- Next milestone is to add and compare a real trained-adapter row under this same trust-first rubric.",
    ]
)

takeaway_text = "\n".join(takeaway_lines)
display(Markdown(takeaway_text))

(root / 'artifacts' / 'tables').mkdir(parents=True, exist_ok=True)
Path(root / 'artifacts' / 'tables' / 'final_reader_takeaways.md').write_text(
    takeaway_text + "\n", encoding='utf-8'
 )
print('Wrote artifacts/tables/final_reader_takeaways.md')

## 7) Reader takeaway summary

This section converts the latest delta table into plain-language findings.

- Highest delta_grounded_trust_score (including diagnostics): **oracle-upper-bound** ($\Delta$ = 0.8343).
- Recommended practical variant today (leaderboard gate-pass): **grounded-heuristic-v1** ($\Delta$ trust = 0.6865, $\Delta$ F1 = 0.2256, $\Delta$ abstain = 0.2069, $\Delta$ citation precision = 1.0000, $\Delta$ citation recall = 1.0000).
- Variants with simultaneous gains on grounding and answer quality: oracle-upper-bound, grounded-heuristic-v1.

### Where we stand
- The evaluation loop is reproducible end-to-end and now tracks a composite grounded trust score.
- Practical ranking is now artifact-backed via finetune_variant_leaderboard outputs.
- Next milestone is to add and compare a real trained-adapter row under this same trust-first rubric.

Wrote artifacts/tables/final_reader_takeaways.md


## 8) Interpretation of Current Results

The evaluation stack is now behaving like a dependable decision system rather than a one-off experiment. Core eval, ablations, finetune comparisons, and artifact contract checks all run in sequence and agree with each other, which means we can trust the direction of change from run to run.

The latest trained adapter entry shows that the model is learning to ground answers more consistently than the baseline. Citation precision and citation recall both move up, and the composite trust metric also improves. That is meaningful because it confirms the adapter is not merely memorizing style; it is increasing evidence alignment.

At the same time, the current adapter still trails the leading practical variant on answer quality, and the strict promotion gate correctly keeps it out of the top practical slot for now. This is useful feedback: the system is sensitive enough to reward stronger grounding while still protecting output quality standards.

In practical terms, the project is in a strong iteration loop:
- Infrastructure is stable and reproducible.
- Retrieval-faithfulness gains from adapter training are already visible.
- The next tuning cycle can focus narrowly on recovering answer-token F1 while keeping the grounding gains.